# 手撕DeepSeek MLA

不考虑效率

仅复现原论文公式

> ref: DeepSeek-V2: A Strong, Economical, and Efficient Mixture-of-Experts Language Model

# setting

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [2]:
# @dataclass
class ModelArgs:
    dim: int = 64
    n_heads: int = 8
    n_kv_heads: int =  2

    # down 两者远小于 dim
    dc_kv: int = 4 
    dc_q: int = 4

bs = 3
seq_len = 5

config = ModelArgs()
print(config)

<__main__.ModelArgs object at 0x117c63f90>

In [3]:
h = torch.randn(bs, seq_len, config.dim)

## Standard Multi-Heads Attention

参考Llama3-GQA， 去除rope简易实现

In [4]:
# ref : Llama3 GQA, ./notebook/Llama3-GQA.ipynb
def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    """torch.repeat_interleave(x, dim=2, repeats=n_rep)"""
    bs, slen, n_kv_heads, head_dim = x.shape
    if n_rep == 1:
        return x
    return (
        x[:, :, :, None, :]
        .expand(bs, slen, n_kv_heads, n_rep, head_dim) # 
        .reshape(bs, slen, n_kv_heads * n_rep, head_dim)
    )

class MultiHeadsAttention(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.n_heads = args.n_heads
        self.n_kv_heads = args.n_heads if args.n_kv_heads is None else args.n_kv_heads
        self.head_dim = args.dim // args.n_heads # 18/6 = 3
        self.n_rep = self.n_heads // self.n_kv_heads

        self.wq = nn.Linear(in_features=args.dim, out_features=args.n_heads * self.head_dim,bias=False,)
        self.wk = nn.Linear(in_features=args.dim, out_features=args.n_kv_heads * self.head_dim,bias=False,)
        self.wv = nn.Linear(in_features=args.dim, out_features=args.n_kv_heads * self.head_dim,bias=False,)
        self.wo = nn.Linear(in_features=args.n_heads * self.head_dim, out_features=args.dim,bias=False,)
        
    def forward(
        self,
        x: torch.Tensor,
    ):
        bsz, seqlen, _ = x.shape
        xq, xk, xv = self.wq(x), self.wk(x), self.wv(x)

        # here we ignore RoPE

        xq = xq.view(bsz, seqlen, self.n_heads, self.head_dim)
        xk = xk.view(bsz, seqlen, self.n_kv_heads, self.head_dim)
        xv = xv.view(bsz, seqlen, self.n_kv_heads, self.head_dim)
        
        keys = repeat_kv( xk, self.n_rep )  
        values = repeat_kv( xv, self.n_rep )  
        
        xq = xq.transpose(1, 2)  # (bs, n_local_heads, seqlen, head_dim)
        keys = keys.transpose(1, 2)  # (bs, n_local_heads, seqlen, head_dim)
        values = values.transpose(1, 2)  # (bs, n_local_heads, seqlen, head_dim)

        
        scores = xq @ keys.transpose(2, 3) / math.sqrt(self.head_dim)
        scores = F.softmax(scores.float(), dim=-1).type_as(xq)
        output = scores @ values # (bs, n_local_heads, seqlen, head_dim)
        output = output.transpose(1, 2).contiguous().view(bsz, seqlen, -1)
        mha_output = self.wo(output)
        return mha_output

In [5]:
mha = MultiHeadsAttention(config)
print(mha)

MultiHeadsAttention(
  (wq): Linear(in_features=64, out_features=64, bias=False)
  (wk): Linear(in_features=64, out_features=16, bias=False)
  (wv): Linear(in_features=64, out_features=16, bias=False)
  (wo): Linear(in_features=64, out_features=64, bias=False)
)

In [6]:
out = mha(h)
print(h.shape)
print(out.shape)

torch.Size([3, 5, 64])

torch.Size([3, 5, 64])

## Multi-Heads Latent Attention

### model

1. 以下用down和up权重矩阵，代替直接的wq,wk,wv
2. MLA矩阵发生于训练之时

In [7]:
class MultiHeadsLatentAttention(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.n_heads = args.n_heads
        self.dim = args.dim
        # self.n_kv_heads = args.n_heads if args.n_kv_heads is None else args.n_kv_heads
        self.head_dim = args.dim // args.n_heads # 18/6 = 3
        self.dc_kv = args.dc_kv
        self.dc_q = args.dc_q

        # MLA Structure
        self.wq_down = nn.Linear(in_features=args.dim, out_features=args.dc_q, bias=False,)
        self.wq_up = nn.Linear(in_features=args.dc_q, out_features=args.dim , bias=False,)

        self.wkv_down = nn.Linear(in_features=args.dim, out_features=args.dc_kv, bias=False,)
        self.wk_up = nn.Linear(in_features=args.dc_kv, out_features=args.dim, bias=False,)
        self.wv_up = nn.Linear(in_features=args.dc_kv, out_features=args.dim, bias=False,)
        
        self.wo = nn.Linear(in_features=args.dim, out_features=args.dim,bias=False,)


In [8]:
mla = MultiHeadsLatentAttention(config)
print(mla)

MultiHeadsLatentAttention(
  (wq_down): Linear(in_features=64, out_features=4, bias=False)
  (wq_up): Linear(in_features=4, out_features=64, bias=False)
  (wkv_down): Linear(in_features=64, out_features=4, bias=False)
  (wk_up): Linear(in_features=4, out_features=64, bias=False)
  (wv_up): Linear(in_features=4, out_features=64, bias=False)
  (wo): Linear(in_features=64, out_features=64, bias=False)
)

### forward

In [9]:
# 该段代码是关键，先降维，再升维
# xq = mha.wq(h)
c_q = mla.wq_down(h)
xq = mla.wq_up(c_q)

# xk = mha.wk(h)
# xk = mha.wv(h)
c_kv = mla.wkv_down(h)
xk = mla.wk_up(c_kv)
xv = mla.wv_up(c_kv)

print(xq.shape)
print(xk.shape)
print(xv.shape)

torch.Size([3, 5, 64])

torch.Size([3, 5, 64])

torch.Size([3, 5, 64])

In [10]:
# 与传统多头注意力无差别
xq = xq.view(bs, seq_len, mla.n_heads, mla.head_dim)
xk = xk.view(bs, seq_len, mla.n_heads, mla.head_dim)
xv = xv.view(bs, seq_len, mla.n_heads, mla.head_dim)

query = xq.transpose(1,2)
key = xk.transpose(1,2) # bs, n_heads, [seq_len, head_dim]
value = xv.transpose(1,2)
print(query.shape) 
print(key.shape)

scores = query @ key.transpose(2,3) / math.sqrt(mla.head_dim) # keys: bs, n_heads, [head_dim, seq_len]
scores = F.softmax(scores.float(), dim=-1).type_as(xq)
output = scores @ value 
output = output.transpose(1, 2).contiguous().view(bs, seq_len, -1)
output = mla.wo(output)
print(output)

torch.Size([3, 8, 5, 8])

torch.Size([3, 8, 5, 8])

tensor([[[-6.3149e-03, -3.3129e-02,  4.2999e-02, -3.8568e-03,  6.3540e-03,
           2.9465e-02, -2.3351e-02, -5.1405e-02,  2.8668e-03,  2.2180e-02,
          -1.9385e-02, -4.2788e-03,  4.1360e-02, -3.5057e-02,  2.0974e-02,
          -2.0437e-02,  4.3493e-03,  1.1378e-02,  8.1210e-03,  1.4886e-02,
           6.6740e-03, -8.2365e-03,  3.2600e-02,  3.0570e-02,  1.2236e-02,
          -3.2159e-02,  2.8724e-02,  2.3854e-02,  3.7415e-02,  1.3003e-02,
           2.4078e-02, -4.2715e-02, -1.2536e-02,  5.2147e-03, -4.7873e-03,
          -2.4776e-02,  2.5847e-02, -2.9202e-02,  2.9333e-02,  4.2691e-03,
           2.2381e-02, -4.7735e-02, -7.7927e-03, -3.9251e-02,  1.4118e-02,
           7.1490e-02, -2.1688e-02, -8.5189e-02,  3.3154e-03,  5.2129e-04,
          -1.3466e-02, -8.5151e-03,  2.7828e-03, -5.6225e-02, -1.7874e-03,
           4.7740e-02,  4.9968e-02, -1.3473e-02,  1.6697e-02,  5.0036e-02,
           1.9111e-02, -4.9846e-02, -9.2807e-03,  9.4121e-03],
         [-7.7569e-02, -2.8469e-02,  1.2605e-02, -3.8820e-02, -1.7759e-02,
           1.3987e-02, -7.3184e-02, -8.4967e-02,  1.7720e-03,  4.0818e-02,
          -1.3131e-02,  4.9411e-02,  5.1238e-02, -5.6356e-02, -6.2477e-02,
          -6.6129e-02, -4.4809e-03,  5.6085e-02,  3.1731e-02,  2.7682e-02,
          -1.4648e-02,  1.9524e-02,  8.1945e-03, -1.8965e-02, -4.7868e-03,
          -3.5099e-02,  4.5350e-02,  8.5431e-03,  7.0573e-02,  9.4237e-03,
          -2.0032e-02, -1.8466e-02, -4.1249e-03, -5.7524e-02, -1.4748e-02,
           1.2206e-02,  3.6227e-03, -3.1404e-02, -1.5336e-02, -1.3845e-03,
           6.5350e-02,  1.1499e-02, -2.8174e-02,  2.4327e-03,  1.8507e-02,
           6.7638e-02, -2.4609e-02, -1.0728e-01, -7.4432e-03,  5.2224e-02,
           6.0145e-04, -3.6431e-02,  4.0157e-02, -5.3719e-02,  4.1967e-02,
          -9.2250e-03,  3.0584e-02, -3.9701e-02,  6.5699e-02,  7.8823e-02,
           1.5196e-02, -8.3512e-02,  4.2717e-02,  3.2396e-02],
         [-1.4648e-01, -5.8195e-02,  5.8918e-02, -1.8603e-02, -7.2529e-03,
           5.2655e-02, -5.7979e-02, -8.0775e-02, -1.2705e-02,  9.8872e-02,
           1.6653e-02,  6.3030e-02,  2.4598e-02, -6.1041e-02, -3.7710e-02,
          -1.1715e-01,  3.0413e-02,  3.5934e-02, -6.9385e-02,  2.1259e-02,
           1.6653e-02,  5.5044e-02,  5.4260e-02, -5.5661e-02,  6.8633e-02,
           4.1733e-02,  5.1267e-02,  8.7736e-03,  7.6374e-02, -5.1411e-03,
          -5.0130e-02, -3.9316e-02, -8.5957e-03, -1.2952e-02,  8.0699e-02,
           1.6721e-02, -3.5053e-02, -7.3948e-02, -6.6606e-02, -1.7816e-02,
           9.7078e-02,  5.9213e-02, -1.9169e-02, -5.9409e-02,  6.8641e-03,
           6.1805e-02, -3.1681e-02, -5.2502e-02,  1.5177e-02,  2.6501e-02,
          -6.5188e-02, -2.5137e-02,  1.1889e-01, -3.6895e-03, -8.3597e-03,
          -5.0518e-02, -2.0848e-02, -1.0968e-01,  2.7654e-02,  3.8654e-02,
          -9.2535e-03, -5.1094e-02,  6.5324e-02,  9.4866e-03],
         [-8.3203e-02, -3.9846e-02,  1.1361e-02, -4.1773e-02, -8.5115e-03,
           2.1091e-02, -6.7009e-02, -9.6057e-02,  9.5003e-03,  4.9302e-02,
          -3.8457e-03,  5.5517e-02,  5.5067e-02, -5.1265e-02, -6.3660e-02,
          -7.3896e-02, -6.6075e-03,  4.5796e-02,  2.1732e-02,  1.8914e-02,
          -1.4420e-02,  2.0478e-02,  7.6116e-03, -2.5418e-02,  2.0799e-05,
          -2.2929e-02,  4.4761e-02,  1.5900e-02,  8.2089e-02, -4.4149e-05,
          -2.1467e-02, -2.0430e-02, -7.7499e-03, -4.8571e-02,  1.0437e-03,
           1.8530e-02,  2.0585e-03, -2.5675e-02, -1.6181e-02, -8.8161e-04,
           7.4571e-02,  1.6692e-02, -2.1747e-02,  4.3380e-03,  2.2727e-02,
           6.8324e-02, -3.0526e-02, -9.7847e-02, -8.7666e-03,  4.8154e-02,
           3.3218e-03, -3.2764e-02,  4.6547e-02, -5.7747e-02,  3.6123e-02,
          -1.1362e-02,  1.5046e-02, -4.5396e-02,  4.1444e-02,  7.9713e-02,
           9.7886e-03, -7.1543e-02,  5.3930e-02,  2.8983e-02],
         [-9.4771e-03, -2.4366e-02,  3.3202e-02, -1.3969e-02, -3.7943e-03,
           1.7408e-02, -3.6333e-02, -5.2228e-02,  1.6773e-03,  1.5169e-02

### 矩阵吸收

以上参数发生在训练之时，训练完成后，我们可以做吸收操作，具体指wq参数和Wo参数，

我们写出如下等式：

Q = wq_up * ( wq_down * h ) = (wq_up * wq_down) * h

Q = wq * h 

目的是什么？

1. 训练时省显存
2. 训练完，推理Q满矩阵保精度，
3. 由于KV Cache的存在，decoding阶段时one-by-one token进行q计算
4. KV Cache极具减少存储体现。

In [11]:
wq = mla.wq_up.weight.data @ mla.wq_down.weight.data 

### 矩阵吸收后的forward(非训练阶段）

In [12]:
# 该段代码是关键，先降维，再升维
xq =  h @ wq
# c_q = mla.wq_down(h) #去除
# xq = mla.wq_up(c_q) #去除

c_kv = mla.wkv_down(h)
xk = mla.wk_up(c_kv)
# xv = mla.wv_up(c_kv) # 去除

# 与传统多头注意力无差别
xq = xq.view(bs, seq_len, mla.n_heads, mla.head_dim)
xk = xk.view(bs, seq_len, mla.n_heads, mla.head_dim)
xv = xv.view(bs, seq_len, mla.n_heads, mla.head_dim)

query = xq.transpose(1,2)
key = xk.transpose(1,2) # bs, n_heads, [seq_len, head_dim]
value = xv.transpose(1,2)
print(query.shape) 
print(key.shape)

# keys: bs, n_heads, [head_dim, seq_len]
scores = query @ key.transpose(2,3) / math.sqrt(mla.head_dim) 
scores = F.softmax(scores.float(), dim=-1).type_as(xq)
output = scores @ value 
output = output.transpose(1, 2).contiguous().view(bs, seq_len, -1)
output = mla.wo(output)
print(output)

torch.Size([3, 8, 5, 8])

torch.Size([3, 8, 5, 8])

tensor([[[-6.8243e-02, -8.9958e-02,  2.4597e-02, -8.5113e-02,  1.1496e-03,
          -3.1157e-02, -7.2015e-02, -9.0081e-02,  2.0451e-02,  6.0989e-02,
          -4.0627e-03,  4.8928e-02,  2.1598e-03, -6.7705e-02, -1.3021e-01,
          -7.9444e-02,  5.7753e-02,  1.8369e-02,  5.9819e-02, -1.0752e-02,
          -3.2157e-02,  3.5679e-02,  5.3930e-02, -3.4948e-03,  3.6266e-02,
           7.0799e-05,  3.8295e-02,  9.0503e-02,  7.0048e-02, -1.7889e-02,
          -3.6864e-02, -1.8640e-02,  4.0646e-02, -6.1394e-02, -1.9804e-02,
           4.3040e-02,  5.0873e-02, -1.5912e-02, -3.2217e-02, -3.4246e-02,
           1.3013e-01, -8.9684e-03,  8.5101e-03,  5.8099e-02,  1.7105e-03,
           5.7786e-02, -2.3046e-03, -8.6906e-02,  3.4365e-02, -3.1938e-03,
           2.6578e-02, -2.2105e-02,  2.3230e-02, -7.0651e-02,  1.4921e-02,
           2.2229e-02,  8.6552e-02, -1.4896e-02,  2.5742e-02,  9.0568e-02,
          -6.0993e-03, -1.0094e-01,  1.3297e-01,  7.7800e-02],
         [-4.8217e-02, -2.9517e-02,  3.9200e-02, -2.8028e-02, -6.9838e-03,
           6.0592e-02, -3.5779e-02, -7.1572e-02,  1.0513e-02,  3.9819e-02,
          -9.9368e-03,  4.8865e-02,  4.1354e-02, -3.9622e-02,  2.3914e-02,
          -4.3236e-02,  9.3757e-03,  2.6543e-02, -8.0670e-03,  1.7248e-02,
           6.3112e-03, -1.3610e-02,  1.2297e-02, -1.4317e-02,  7.6450e-03,
          -2.4071e-02,  4.9910e-02,  1.1014e-02,  5.6355e-02,  4.4315e-03,
           3.1770e-02, -2.2386e-02, -8.2471e-03, -6.4493e-03,  1.6498e-02,
          -2.7831e-02,  7.8901e-03, -4.1972e-02, -1.7499e-02, -1.4566e-02,
           3.2114e-02,  2.3054e-04, -2.6391e-02, -2.0519e-02,  9.8399e-03,
           6.4161e-02, -2.4791e-02, -8.5359e-02, -2.7361e-02,  2.5472e-02,
          -3.4014e-02, -3.2157e-02,  3.7375e-02, -5.1151e-02,  2.5577e-02,
           5.1952e-03,  9.1408e-03, -2.4076e-02,  3.3693e-02,  6.2376e-02,
           9.1614e-03, -8.0952e-02, -5.1676e-03,  1.0061e-02],
         [-3.2740e-02,  1.4279e-02,  3.4701e-02,  2.4030e-02, -2.0018e-02,
           1.0815e-01, -1.1087e-02, -5.0999e-02,  3.5169e-03,  1.4644e-02,
          -4.8108e-03,  4.8569e-02,  6.8108e-02,  1.4916e-03,  1.0979e-01,
          -1.2142e-02, -3.0312e-02,  3.4281e-02, -5.0458e-02,  3.9530e-02,
           1.7778e-02, -4.1841e-02, -3.6164e-02, -2.0637e-02, -3.7623e-02,
          -4.6141e-02,  4.6984e-02, -4.5600e-02,  6.2120e-02,  2.1584e-02,
           5.6616e-02, -3.5344e-02, -4.6860e-02,  3.3107e-02,  3.1111e-02,
          -5.7685e-02, -2.4973e-02, -5.0550e-02,  1.3018e-02, -1.6972e-03,
          -4.5431e-02, -8.7277e-04, -4.1318e-02, -6.6588e-02,  2.0440e-02,
           4.9733e-02, -4.2404e-02, -7.7699e-02, -6.0576e-02,  5.3518e-02,
          -7.0500e-02, -4.8211e-02,  4.1884e-02, -2.4815e-02,  2.6454e-02,
          -3.8905e-03, -6.4834e-02, -3.5008e-02,  3.8153e-02,  4.4375e-02,
           3.1449e-02, -2.9374e-02, -1.0395e-01, -4.7478e-02],
         [-4.8922e-02, -2.2901e-02,  2.1123e-02, -2.0936e-02, -9.0501e-03,
           4.5224e-02, -3.5751e-02, -7.1526e-02,  1.3995e-02,  2.4621e-02,
          -7.5754e-03,  5.2715e-02,  5.1554e-02, -2.3388e-02,  8.0347e-03,
          -3.6891e-02,  1.1322e-03,  2.8618e-02, -5.8978e-03,  2.4045e-02,
          -2.3311e-02, -7.3189e-03, -5.7786e-03, -1.6993e-02, -2.5943e-02,
          -2.9507e-02,  4.2561e-02,  5.6950e-03,  6.9569e-02,  1.0374e-02,
           1.0157e-02, -2.5334e-02, -1.4184e-02, -7.9973e-04,  5.3356e-03,
          -1.5590e-02,  1.0197e-02, -2.8491e-02,  4.1600e-03, -1.6092e-02,
           2.1452e-02, -4.8319e-03, -2.6199e-02, -1.4409e-02,  1.2900e-02,
           5.0491e-02, -2.2232e-02, -9.0220e-02, -1.7879e-02,  3.9548e-02,
          -2.3741e-02, -3.8930e-02,  2.3561e-02, -4.2296e-02,  2.4941e-02,
           1.1671e-02, -7.5167e-03, -2.1648e-02,  3.6893e-02,  7.0894e-02,
           2.0150e-02, -4.5152e-02, -6.3353e-03,  6.0141e-03],
         [-7.0406e-02, -6.8714e-02,  4.2928e-02, -8.0124e-02, -2.2692e-02,
           5.0637e-02, -5.8464e-02, -9.0205e-02,  1.8653e-02,  7.7669e-02

In [13]:
import torch

# 假设 tensor 是形状为 (a, b, c) 的张量
tensor = torch.randn(2, 3, 4)

# 扩展第 1 维，复制 8 份
expanded_tensor = tensor.repeat(8, 1, 1)
print(expanded_tensor.shape)

torch.Size([16, 3, 4])

### KV cache存的是什么？

传统MHA的KV Cache大小: `[2, bs, seq_len, n_kv_head * head_dim]`, 其中2代表K和V

如果我们存储MLA up后的矩阵K,V cache，那么与MHA存储无差别

那么我们可以存储kv down的矩阵：即 c_kv = w_kv_down @ h, 此时：

`[1, bs, seq_len, dc_kv]`

如果原来  2 * n_kv_head * head_dim = dim = 2 * 4096, 如果dc_kv  = 512

那么MLA KV-cache就为MHA的 1 / 16

### MLA压缩的本质是什么？

计算时间换空间

存储时刻，w_k_up

decoding时刻 即将：

`K = w_k_up @ c_kv`

`V = w_v_up @ c_kv`

压缩KV-Cache的量的意义？以VLLM来说，减少KV-Cache量。推理服务能跑更大的batch-size，从而提高inference，decoding的效率

### MLA下位置编码问题

1. 常规算法为：RoPE(W_k_up( w_k_down(h) )), 这里的RoPE没法低秩分解
2. 可以写为attention： `Rk @ Wkup @ wkdown(h)^T` `@` `[Rq @ Wqup @ wqdown(h)]^T` 
3. 上式注意到，我们所存储kv cache是wkdown(h)为 `seq x dc_kv`, 那么我们每次计算出了要up，而且还要对**所有token**增加RoPE操作
4. 有什么方式可以减少RoPE操作吗？解决方案为在h上，增加额外的参数矩阵比如：

In [14]:
class MultiHeadsLatentAttention_withRoPE(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.n_heads = args.n_heads
        self.dim = args.dim
        # self.n_kv_heads = args.n_heads if args.n_kv_heads is None else args.n_kv_heads
        self.head_dim = args.dim // args.n_heads # 18/6 = 3
        self.dc_kv = args.dc_kv
        self.dc_q = args.dc_q

        # MLA Structure
        self.wq_down = nn.Linear(in_features=self.dim, out_features=self.dc_q, bias=False,)
        self.wq_up = nn.Linear(in_features=self.dc_q, out_features=self.dim , bias=False,)

        self.wkv_down = nn.Linear(in_features=self.dim, out_features=self.dc_kv, bias=False,)
        self.wk_up = nn.Linear(in_features=self.dc_kv, out_features=self.dim, bias=False,)
        self.wv_up = nn.Linear(in_features=self.dc_kv, out_features=self.dim, bias=False,)
        
        self.wo = nn.Linear(in_features=self.dim, out_features=self.dim,bias=False,)

        # RoPE Weight
        # K 每头一样， Q每头不一样
        self.wq_up_rope = nn.Linear(in_features=self.dc_q, out_features=self.dim , bias=False,)
        self.wk_head_rope = nn.Linear(in_features=self.dim, out_features=self.head_dim , bias=False,)

In [15]:
mla_rope = MultiHeadsLatentAttention_withRoPE(config)
print(mla_rope)

MultiHeadsLatentAttention_withRoPE(
  (wq_down): Linear(in_features=64, out_features=4, bias=False)
  (wq_up): Linear(in_features=4, out_features=64, bias=False)
  (wkv_down): Linear(in_features=64, out_features=4, bias=False)
  (wk_up): Linear(in_features=4, out_features=64, bias=False)
  (wv_up): Linear(in_features=4, out_features=64, bias=False)
  (wo): Linear(in_features=64, out_features=64, bias=False)
  (wq_up_rope): Linear(in_features=4, out_features=64, bias=False)
  (wk_head_rope): Linear(in_features=64, out_features=8, bias=False)
)

In [16]:
# 增加rope
c_q = mla_rope.wq_down(h)
xq = mla_rope.wq_up(c_q)

c_kv = mla_rope.wkv_down(h)
xk = mla_rope.wk_up(c_kv)
xv = mla_rope.wv_up(c_kv)

# print(xq.shape)
# print(xk.shape)
# print(xv.shape)

# 位置编码相关
r_q = mla_rope.wq_up(c_q) #多头
r_k = mla_rope.wk_head_rope(h) #单头
print(r_q.shape)
print(r_k.shape)

torch.Size([3, 5, 64])

torch.Size([3, 5, 8])

产生新的疑问，r_q 和 r_k 维度不同，那么做rope的dim如何处理？

In [17]:
# 简易写下
rope_matrix_q = [torch.randn(mla_rope.dim, mla_rope.dim)] * seq_len
rope_matrix_k = [torch.randn(mla_rope.head_dim, mla_rope.head_dim)] * seq_len
def apply_rope_q(x, seq_len, rope_matrix):
    for i in range(seq_len):
        x[:, i, :] = x[:, i, :] @ rope_matrix[i]
    return x

rope_q = apply_rope_q(r_q, seq_len, rope_matrix_q)
rope_k = apply_rope_q(r_k, seq_len, rope_matrix_k)

对于q每头，cat不一样的位置信息

对于k每头，cat一样的信息

In [18]:
# 与传统多头注意力无差别
xq = xq.view(bs, seq_len, mla_rope.n_heads, mla_rope.head_dim)
xk = xk.view(bs, seq_len, mla_rope.n_heads, mla_rope.head_dim)
xv = xv.view(bs, seq_len, mla_rope.n_heads, mla_rope.head_dim)


query = xq.transpose(1,2)
key = xk.transpose(1,2) # bs, n_heads, [seq_len, head_dim]
value = xv.transpose(1,2)

print(query.shape) 
print(key.shape)

## 嵌入rope
rope_q_head = rope_q.view(bs, seq_len, mla_rope.n_heads, mla_rope.head_dim)
rope_q_head = rope_q_head.transpose(1,2)

rope_k_head = rope_k.unsqueeze(dim = 1).repeat( repeats = [1, mla_rope.n_heads, 1, 1])
print(rope_q_head.shape)
print(rope_k_head.shape)


# cat 操作
query_cat = torch.cat((query, rope_q_head), dim = -1)
key_cat = torch.cat((query, rope_k_head), dim = -1)

torch.Size([3, 8, 5, 8])

torch.Size([3, 8, 5, 8])

torch.Size([3, 8, 5, 8])

torch.Size([3, 8, 5, 8])

In [19]:
### 常规attention
# scores = query @ key.transpose(2,3) # keys: bs, n_heads, [head_dim, seq_len]
# keys: bs, n_heads, [2head_dim, seq_len]
scores = query_cat @ key_cat.transpose(2,3) / math.sqrt(2 * mla_rope.head_dim) # cat后dim维度变化，底数也有变化 
scores = F.softmax(scores.float(), dim=-1).type_as(xq)
output = scores @ value 
output = output.transpose(1, 2).contiguous().view(bs, seq_len, -1)
output = mla.wo(output)
print(output.shape)

torch.Size([3, 5, 64])

## 增加问题

1. 为什么位置编码要分离
2. q和k的位置编码维度不一样，那么rope的维度是否一样
3. v_up 如何被 wo 吸收
5. 写出带kv_cache版本的MLA，及decoding代码
6. MLA训练的显存计算
7. MLA并行参数分配、张量并行细节和通信